# Redrob Ranker — Demo Sandbox

This notebook demonstrates the **Redrob Intelligent Candidate Discovery** ranker on a 50-candidate sample.

It clones the repo, installs dependencies, and runs the full ranking pipeline end-to-end — CPU only, no network calls during ranking, no GPU required.

**Runtime:** ~15 seconds for 50 candidates. Full 100K run takes ~90s on CPU.

In [ ]:
# Step 1: Clone the repo
!git clone https://github.com/spg3098-alt/redrob-ranker.git
%cd redrob-ranker

In [ ]:
# Step 2: Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Step 3: Run the ranker on the 50-candidate demo sample
# sample_candidates.jsonl is pre-bundled in the repo
!python rank.py --candidates ./sample_candidates.jsonl --out ./demo_output.csv

In [ ]:
# Step 4: Show results
import pandas as pd
df = pd.read_csv('demo_output.csv')
print(f'Ranked {len(df)} candidates')
print()
pd.set_option('display.max_colwidth', 120)
print(df[['rank', 'candidate_id', 'score', 'reasoning']].to_string(index=False))

## How it works

```
additive   = 0.22*title + 0.38*skills_trust + 0.14*semantic
           + 0.10*ml_yoe_experience + 0.08*location + 0.08*eval_text_evidence

gated      = additive
           × domain_mismatch_penalty
           × consulting_only_penalty
           × notice_multiplier          # x0.65-1.00 by notice days
           × ml_yoe_gate                # x0.25/<3y, x0.60/<4y, x0.95/<5y
           × core_coverage_gate         # x0.70 if <=1/4 must-haves covered
           × self_assessment_penalty    # x0.85 if summary admits weakness

final      = gated
           x behavioral_multiplier      # x0.40-1.0 (capped for low coverage)
           x honeypot_sink              # x0.03 for impossible profiles
```

See `src/redrob_ranker/` for full source. No LLM calls, no GPU, no network during ranking.